# Template de medición OIL con `oscrigol_oil`
Plantilla genérica para capturar señales OIL usando el wrapper basado en `oscrigol.py`.


## 1. Imports
Carga utilidades de plotting y las funciones `oil_med` / `plotresults`.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "oscrigol_oil.py").exists():
    HERE = Path("oscilloscopes/rigol-MSO2102A").resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from oscrigol_oil import oil_med, plotresults


## 2. Parámetros de adquisición
Define cantidad de mediciones, promediado, memoria y backend VISA.


In [ ]:
NMED = 10
ACQ = 1
MDEPTH = 70000
SAVE_RESULTS = True
OUTPUT_DIR = "./Mediciones/"

IP_ADDRESS = "192.168.2.2"
TRANSPORT = "socket"
VISA_BACKEND = "pyvisa"  # Cambiar a "nivisa" si se quiere usar NI-VISA.
DOWNLOAD_MODE = "fast"

WAVEFORM_POINTS_TIMEOUT = 0.5
MAX_RETRIES = 2
MIN_VPP = None


## 3. Ejecutar medición
Llama a `oil_med()` y devuelve tiempo, matriz de señales, energía y variables ambientales.


In [ ]:
t, MV, E, T = oil_med(
    Nmed=NMED,
    acq=ACQ,
    mdepth=MDEPTH,
    saveresults=SAVE_RESULTS,
    output_dir=OUTPUT_DIR,
    ip_address=IP_ADDRESS,
    transport=TRANSPORT,
    visa_backend=VISA_BACKEND,
    download_mode=DOWNLOAD_MODE,
    waveform_points_timeout=WAVEFORM_POINTS_TIMEOUT,
    max_retries=MAX_RETRIES,
    min_vpp=MIN_VPP,
)


## 4. Inspección rápida
Muestra dimensiones y un resumen básico de energía/tiempo.


In [ ]:
print("t:", np.shape(t), "MV:", np.shape(MV), "E:", np.shape(E), "T:", np.shape(T))
if len(E):
    print("E media [mJ]:", np.nanmean(E) * 1e3)
if len(T):
    print("duración [min]:", np.nanmax(T[:, 0]) / 60)


## 5. Graficar resultados estándar
Usa `plotresults()` para revisar señales, energía y variables ambientales.


In [ ]:
plotresults(t, MV, E, T)


## 6. Superponer señales
Grafica todas las mediciones en una sola figura para detectar outliers.


In [ ]:
plt.figure(figsize=(9, 4))
for row in MV:
    plt.plot(np.asarray(t) * 1e6, row * 1e3, alpha=0.5)
plt.xlabel("tiempo [us]")
plt.ylabel("amplitud [mV]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.title("Señales OIL")


## 7. Señal promedio
Calcula y grafica la señal promedio de la tanda.


In [ ]:
mean_signal = np.nanmean(MV, axis=0)
plt.figure(figsize=(9, 4))
plt.plot(np.asarray(t) * 1e6, mean_signal * 1e3)
plt.xlabel("tiempo [us]")
plt.ylabel("amplitud media [mV]")
plt.grid(True, linestyle="--", alpha=0.5)
plt.title("Promedio OIL")
